# Load necessary models

In [1]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27018/")
db = client["picam"]
collection = db["images"]

In [2]:
import clip
import torch
checkpoint_path = "conclip_vit_l14.pt"
device = "cuda"

# the .pt file downloaded from the links above
device = "cuda"
def load_checkpoint(model, checkpoint_path):
	ckpt = torch.load(checkpoint_path, weights_only=False)
	model = model.float()
	model.load_state_dict(ckpt["model"])
	return model

model, preprocess = clip.load("ViT-L/14", device="cuda")
model = load_checkpoint(model, "conclip_vit_l14.pt")
model.eval()
model = model.to(device)

FileNotFoundError: [Errno 2] No such file or directory: 'conclip_vit_l14.pt'

In [ ]:
def predict(texts, features):
    with torch.no_grad():
        text_inputs = clip.tokenize(texts).to(device)
        text_features = model.encode_text(text_inputs)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        similarity = (text_features @ features.T).cpu().numpy()
        return similarity

# Step 1: Assign movement to images

In [ ]:
FEATURE_DIR = "/mnt/ssd0/embeddings/conclip_vit_l14/cathal_features/"

In [ ]:
moves = {"I am sitting on an airplane": "Airplane",
         "I am in a car": "Car",
         "I am in an airport": "Inside",
         "I am cycling": "Cycling",
         "I am walking outside or on the street": "Walking Outside",
         "I am on public transport": "Public Transport",
         "I am inside a building or a house": "Inside"}

insides = {"I am inside a building or a house": "Inside",
           "I am outside": "Outside",
           "I am in a transport": "Transport"}


In [ ]:
import os
from tqdm.auto import tqdm
import numpy as np

from pymongo import UpdateOne
images = sorted(os.listdir(FEATURE_DIR))
# exclude files that start with any of these prefixes
EXCLUDE_PREFIXES = (
    "2016", "2019",
    "202001", "202002", "202003", "202004", "202005", "202006",
)

images = [img for img in images if not img.startswith(EXCLUDE_PREFIXES)]

text_moves = list(moves.keys())
text_insides = list(insides.keys())
texts = text_moves + text_insides
with torch.no_grad():
    text_inputs = clip.tokenize(texts).to(device)
    text_features = model.encode_text(text_inputs)
    text_features /= text_features.norm(dim=-1, keepdim=True)
nm = len(text_moves)

# process this many images per batch
batch_size = 32
for i in tqdm(range(0, len(images), batch_size)):
    batch_images = images[i:i+batch_size]
    batch_features = []
    for image in batch_images:
        if image.endswith(".npy"):
            features = np.load(os.path.join(FEATURE_DIR, image))
            original_name = image.split(".npy")[0]
            batch_features.append((original_name, features))
    
    if batch_features:
        image_names, image_features = zip(*batch_features)
        image_features = np.stack(image_features)
        image_features = torch.tensor(image_features).to(device)
        image_features /= image_features.norm(dim=-1, keepdim=True)
    else:
        continue
   
    # compute similarity in one matrix multiply on device, then move to CPU once
    similarity = (text_features @ image_features.T).cpu().numpy()

    sim_move = similarity[:nm, :]
    sim_inside = similarity[nm:, :]

    # get best labels for each image (vectorized)
    best_move_idxs = np.argmax(sim_move, axis=0)
    best_inside_idxs = np.argmax(sim_inside, axis=0)

    move_labels_list = [moves[text_moves[i]] for i in best_move_idxs]
    inside_labels_list = [insides[text_insides[i]] for i in best_inside_idxs]

    # build bulk ops
    date = image.split("_")[0]
    date =  f"{date[:4]}-{date[4:6]}-{date[6:8]}"
    ops = [
        UpdateOne({"image_path": f"{date}/{name}", "device": "cathal"}, {"$set": {"movement": move_label, "inside_outside": inside_label}})
        for name, move_label, inside_label in zip(image_names, move_labels_list, inside_labels_list)
    ]

    if ops:
        collection.bulk_write(ops)

  0%|          | 0/40050 [00:00<?, ?it/s]

# Step 2: Clustering

In [ ]:
# Run gps_pipeline.py
!python3 gps_pipeline.py

# Step 3: Semantic

In [ ]:
# Run enrich_stops.py
!python3 enrich_stops.py

# Step 4: Visualise

In [ ]:
from generate_day_view import run as generate_day_view
generate_day_view()

In [20]:
def fetch_images(start_ts: float, end_ts: float) -> list:
    images = list(collection.find(
        {"timestamp": {"$gte": start_ts, "$lt": end_ts}, "device": "cathal"},
        {"image_path": 1, "timestamp": 1, "time": 1, "movement": 1, "inside_outside": 1},
        sort=[("timestamp", 1)],
    ))
    return images

In [34]:
import pandas as pd
df = pd.read_csv("files/nominatim_semantic_stops.csv", sep=";")
df

,track_id,start,end,start_ts,end_ts,label,centroid_lat,centroid_lon,centroid_alt,stop_id,place_id,n_points,fsq_place_id,name,categories,prob,parent,parent_id,city,region,country,note,address,timezone,movement,location,info
0,0,20200630_230137,20200630_231314,1.593558e+09,1.593559e+09,1,53.389970,-6.145802,67.579906,stop_0_0,place_0,82,NaN,HOME,NaN,0.0,NaN,NaN,Dublin,"['Dublin', 'Ireland']",Ireland,HOME,HOME,Europe/Dublin,NaN,NaN,NaN
1,1,20200630_233213,20200701_055836,1.593560e+09,1.593583e+09,1,53.389973,-6.145787,68.304904,stop_1_0,place_0,3513,NaN,HOME,NaN,0.0,NaN,NaN,Dublin,"['Dublin', 'Ireland']",Ireland,HOME,HOME,Europe/Dublin,NaN,NaN,NaN
2,2,20200701_065737,20200701_070250,1.593587e+09,1.593587e+09,1,53.385345,-6.141535,57.299606,stop_2_0,place_0,19,4f2c43a26d86f72ff8b5e317,Apache Pizza,Pizzeria,0.0,NaN,NaN,['Dublin'],"['County Dublin', 'Dublin', 'Ireland']",Ireland,default,"12 Dublin Rd, Dublin, County Dublin",Europe/Dublin,Car,"{""address"": ""12 Dublin Rd"", ""locality"": ""Dubli...",NaN
3,2,20200701_070311,20200701_070415,1.593587e+09,1.593587e+09,0,53.388026,-6.128832,67.771136,NaN,NaN,10,NaN,Sutton,NaN,0.0,NaN,NaN,Sutton,Leinster,Ireland,move segment enriched with Nominatim geocoding,NaN,Europe/Dublin,Car,NaN,Car
4,2,20200701_070425,20200701_070609,1.593587e+09,1.593587e+09,1,53.391391,-6.124581,64.144882,stop_2_1,place_2,16,4c2b3f1bb34ad13a2a6ae9ce,Maxol,Fuel Station,0.0,NaN,NaN,['Dublin'],"['County Dublin', 'Dublin', 'Ireland']",Ireland,default,"Baldoyle Rd, Dublin, County Dublin",Europe/Dublin,Car,"{""address"": ""Baldoyle Rd"", ""locality"": ""Dublin...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49182,15554,20221231_132507,20221231_132507,1.672493e+09,1.672493e+09,0,21.023307,105.854576,-13.556277,NaN,NaN,2,NaN,Hà Nội,NaN,0.0,NaN,NaN,Hà Nội,NaN,VN,move segment enriched with Nominatim geocoding,NaN,Asia/Bangkok,NaN,NaN,NaN
49183,15555,20221231_133727,20221231_134430,1.672494e+09,1.672494e+09,1,21.024354,105.854713,-13.774150,stop_15555_0,place_1297,80,5e2ec38f4ce734000c847acd,Artemis - Art Of Pastry,Coffee Shop,0.0,NaN,NaN,['Hà Nội'],"['Thành Phố Hà Nội', 'Hà Nội', 'VN']",VN,default,"20 Ngô Quyền, Hà Nội, Thành Phố Hà Nội",Asia/Bangkok,Inside,"{""address"": ""20 Ng\u00f4 Quy\u1ec1n"", ""localit...",NaN
49184,15556,20221231_141518,20221231_144430,1.672496e+09,1.672498e+09,1,21.028277,105.853629,-13.536447,stop_15556_0,place_1479,514,4bd809a5f645c9b6e605a7e0,Tràng Tiền Plaza,Shopping Mall,0.0,NaN,NaN,['Hà Nội'],"['Thành Phố Hà Nội', 'Hà Nội', 'VN']",VN,default,"1 Tràng Tiền (Hàng Bài), Hà Nội, Thành Phố Hà Nội",Asia/Bangkok,Walking Outside,"{""address"": ""1 Tr\u00e0ng Ti\u1ec1n"", ""localit...",NaN
49185,15557,20221231_145132,20221231_154039,1.672498e+09,1.672501e+09,1,21.035720,105.850366,-12.158815,stop_15557_0,place_1291,476,567d4e89498eec44e9dff649,MK Premier Boutique Hotel,Hotel,0.0,NaN,NaN,"['Hoàn Kiếm', 'Hà Nội', 'Vietnam', 'Hà Nội']","['Thành Phố Hà Nội', 'Hà Nội', 'VN']",VN,default,"72-74, Hoàn Kiếm, Hà Nội, Vietnam, Hà Nội, Thà...",Asia/Bangkok,Walking Outside,"{""address"": ""72-74, Ho\u00e0n Ki\u1ebfm, H\u00...",NaN


In [27]:
from tqdm.auto import tqdm
# assigning movement labels to "categories" by the images
for idx, row in tqdm(moves.iterrows(), total=len(moves)):
    start_ts = row["start_ts"] * 1000
    end_ts = row["end_ts"] * 1000
    images = fetch_images(start_ts, end_ts)
    if not images:
        continue
    
    # get the most common movement label among the images
    movement_labels = [img.get("movement") for img in images if img.get("movement")]
    if movement_labels:
        most_common_label = max(set(movement_labels), key=movement_labels.count)
        df.at[idx, "movement"] = most_common_label

  0%|          | 0/22282 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
unknown = df[df["name"] == "Unknown Place"]
unknown

,track_id,start,end,start_ts,end_ts,label,centroid_lat,centroid_lon,centroid_alt,stop_id,place_id,n_points,fsq_place_id,name,categories,prob,parent,parent_id,location,note
43,27,20200702_094215,20200702_094309,1.593683e+09,1.593683e+09,1,53.776596,-8.380955,166.331201,stop_27_3,place_7,10,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places within distance threshold
45,27,20200702_094502,20200702_094613,1.593683e+09,1.593683e+09,1,53.765198,-8.397167,164.742325,stop_27_4,place_8,12,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places within distance threshold
47,29,20200702_110435,20200702_110936,1.593688e+09,1.593688e+09,1,53.765349,-8.397201,161.298386,stop_29_0,place_8,29,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places within distance threshold
48,30,20200702_111612,20200702_113051,1.593689e+09,1.593689e+09,1,53.765354,-8.397153,167.124544,stop_30_0,place_8,81,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places within distance threshold
51,33,20200702_131146,20200702_132757,1.593696e+09,1.593696e+09,1,53.765384,-8.397274,169.760543,stop_33_0,place_8,84,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places within distance threshold
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49042,15486,20221226_131643,20221226_131708,1.672061e+09,1.672061e+09,1,53.390553,-6.160448,67.301079,stop_15486_276,place_0,11,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places within distance threshold
49050,15492,20221226_192237,20221226_193107,1.672083e+09,1.672083e+09,1,53.389120,-6.157888,75.295257,stop_15492_0,place_0,53,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places found
49072,15509,20221227_172106,20221227_174157,1.672162e+09,1.672163e+09,1,53.394308,-6.163197,78.135800,stop_15509_2,place_0,99,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places found
49074,15509,20221227_174405,20221227_174757,1.672163e+09,1.672163e+09,1,53.389279,-6.158004,68.994718,stop_15509_0,place_0,20,NaN,Unknown Place,NaN,0.0,NaN,NaN,{},No nearby places found


# Step 5: Try Nominatim

In [45]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="lifelog-picam")
location = geolocator.reverse("53.39814400752409, -6.2409560382974965", language="en", exactly_one=False, namedetails=True)

In [46]:
for loc in location:
    print(loc.raw)
    print(loc.address)

{'place_id': 252608913, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 902121700, 'lat': '53.3984163', 'lon': '-6.2403447', 'class': 'building', 'type': 'house', 'place_rank': 30, 'importance': 8.015945677877496e-05, 'addresstype': 'building', 'name': '', 'display_name': '1, Coolock Lane, Santry, Turnapin DED 1986, Fingal, County Dublin, Leinster, D09 X6X4, Ireland', 'address': {'house_number': '1', 'road': 'Coolock Lane', 'suburb': 'Santry', 'city_district': 'Turnapin DED 1986', 'municipality': 'Fingal', 'county': 'County Dublin', 'ISO3166-2-lvl6': 'IE-D', 'region': 'Leinster', 'ISO3166-2-lvl5': 'IE-L', 'postcode': 'D09 X6X4', 'country': 'Ireland', 'country_code': 'ie'}, 'namedetails': None, 'boundingbox': ['53.3983697', '53.3984882', '-6.2403894', '-6.2403010']}
1, Coolock Lane, Santry, Turnapin DED 1986, Fingal, County Dublin, Leinster, D09 X6X4, Ireland


In [33]:
from timezonefinder import TimezoneFinder
tf = TimezoneFinder()
tf.timezone_at(lng=-8.380955, lat=53.776596)

'Europe/Dublin'